<a href="https://colab.research.google.com/github/Renadth/LLM-application-engineering-/blob/main/homework/day2_tools_norah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 Homework — Validate and Route Structured Tool Calls

**Course:** LLM Application Engineering · SDA-AIE-213  
**Instructor:** Meaad Al-Marri  
**Assignment:** Extra homework, Day 2 (Assignment A: controlled tools)  
**Mode:** `simulated_tool_protocol`: Python standard library only. No model, no API key, no package install.

A small dispatcher sits in front of one **read-only** tool, `check_status`. Every structured call is treated as **untrusted input**, even when it looks like valid JSON. The caller's identity comes in **separately** from the call, so nothing inside the call can grant permission.

## Step 1 — Synthetic records and the trusted identity

All data is fictional. `REF-1001` belongs to the session user, `REF-1002` belongs to someone else, and `REF-9999` does not exist (it is used for the not-found test).

In [14]:
import json
import re

# Synthetic records (fictional, no real data)
RECORDS = {
    'REF-1001': {'owner': 'demo-user-1', 'status': 'under_review'},
    'REF-1002': {'owner': 'demo-user-2', 'status': 'approved'},
}

# Trusted demo identity: provided by the "session", never by the tool call
SESSION_USER = 'demo-user-1'

# Explicit allowlist: only one read-only tool is allowed
ALLOWED_TOOLS = {'check_status'}

# Accepted shapes
CALL_KEYS = {'name', 'arguments'}
ARG_KEYS = {'reference'}
REF_PATTERN = re.compile(r'REF-[0-9]{4}')  # ASCII digits only

print('records:', sorted(RECORDS))
print('session user:', SESSION_USER)
print('allowed tools:', sorted(ALLOWED_TOOLS))

records: ['REF-1001', 'REF-1002']
session user: demo-user-1
allowed tools: ['check_status']


## Step 2 — The read-only tool and the dispatcher

The checks run in a fixed order, and each one returns a structured result instead of raising an exception:

1. **Call shape:** must be a dict with exactly `name` and `arguments`, otherwise `invalid_call`
2. **Allowlist:** `name` must be in `ALLOWED_TOOLS`, otherwise `tool_not_allowed`
3. **Arguments:** exactly one key, `reference`, holding a string like `REF-` plus 4 digits, otherwise `invalid_arguments`
4. **Authorization:** the record must exist **and** be owned by the trusted session user, otherwise `unavailable`

A missing record and someone else's record both return `unavailable`, so a caller can't use the reply to learn which references exist.

In [15]:
TOOL_CALLS = {'count': 0}  # counts how many times the real tool actually ran


def check_status(reference, user):
    # Read-only tool: only reads RECORDS, never changes them
    TOOL_CALLS['count'] += 1
    record = RECORDS.get(reference)
    if record is None or record['owner'] != user:
        return {'code': 'unavailable'}          # same answer for missing and not-owned
    return {'code': 'ok', 'status': record['status']}


def dispatch(call, trusted_user):
    # 1) Validate the call structure (untrusted data, never executed)
    if not isinstance(call, dict) or set(call) != CALL_KEYS:
        return {'code': 'invalid_call'}

    # 2) Allowlist check: unknown or mutating tools are rejected
    if call['name'] not in ALLOWED_TOOLS:
        return {'code': 'tool_not_allowed'}

    # 3) Validate arguments: exact keys, type, and format
    args = call['arguments']
    if not isinstance(args, dict) or set(args) != ARG_KEYS:
        return {'code': 'invalid_arguments'}   # e.g. an extra 'owner' field
    ref = args['reference']
    if not isinstance(ref, str) or not REF_PATTERN.fullmatch(ref):
        return {'code': 'invalid_arguments'}   # e.g. a number instead of a string

    # 4) Authorization: identity comes from the session, not from the call
    return check_status(ref, trusted_user)

print('dispatcher ready')

dispatcher ready


## Step 3 — Predict, then run the seven test cases

My prediction for each case, written **before** running:

| # | Input being checked | Predicted result | Why |
|---|---|---|---|
| 1 | Owned reference `REF-1001` | `ok` / `under_review` | Valid call, and the user owns the record |
| 2 | Another user's `REF-1002` | `unavailable` | Valid call, but the ownership check fails |
| 3 | Missing `REF-9999` | `unavailable` | Valid format, but no record exists |
| 4 | Tool `delete_record` | `tool_not_allowed` | Not on the allowlist (mutation attempt) |
| 5 | Extra `owner` argument | `invalid_arguments` | The caller tries to claim an identity |
| 6 | `reference` = `1001` (int) | `invalid_arguments` | Wrong type |
| 7 | A plain string | `invalid_call` | Not a call object at all |

The test also confirms that the records are unchanged after the full run, and that the real tool ran only for cases 1–3.

In [16]:
CASES = [
    ('owned reference',          {'name': 'check_status', 'arguments': {'reference': 'REF-1001'}},                        {'code': 'ok', 'status': 'under_review'}),
    ("another user's reference", {'name': 'check_status', 'arguments': {'reference': 'REF-1002'}},                        {'code': 'unavailable'}),
    ('absent reference',         {'name': 'check_status', 'arguments': {'reference': 'REF-9999'}},                        {'code': 'unavailable'}),
    ('disallowed tool',          {'name': 'delete_record', 'arguments': {'reference': 'REF-1001'}},                       {'code': 'tool_not_allowed'}),
    ('unexpected owner arg',     {'name': 'check_status', 'arguments': {'reference': 'REF-1001', 'owner': 'demo-user-2'}}, {'code': 'invalid_arguments'}),
    ('numeric reference',        {'name': 'check_status', 'arguments': {'reference': 1001}},                              {'code': 'invalid_arguments'}),
    ('string instead of object', 'not a tool object',                                                                      {'code': 'invalid_call'}),
]

snapshot_before = json.dumps(RECORDS, sort_keys=True)
TOOL_CALLS['count'] = 0

rows = []
for i, (label, call, expected) in enumerate(CASES, 1):
    actual = dispatch(call, trusted_user=SESSION_USER)
    ok = actual == expected
    rows.append({'case': i, 'result': actual, 'passed': ok})
    print(f"{i}. {label:<26} -> {json.dumps(actual):<42} {'PASS' if ok else 'FAIL'}")

records_unchanged = json.dumps(RECORDS, sort_keys=True) == snapshot_before
passed = sum(r['passed'] for r in rows)

assert passed == len(CASES), 'a case failed: fix the code, do not weaken the expected result'
assert records_unchanged, 'records were modified'
assert TOOL_CALLS['count'] == 3, 'the tool should only run for the 3 well-formed, allowed calls'

print(f"\npassed {passed}/{len(CASES)} | records unchanged: {records_unchanged} | real tool invocations: {TOOL_CALLS['count']}")

1. owned reference            -> {"code": "ok", "status": "under_review"}   PASS
2. another user's reference   -> {"code": "unavailable"}                    PASS
3. absent reference           -> {"code": "unavailable"}                    PASS
4. disallowed tool            -> {"code": "tool_not_allowed"}               PASS
5. unexpected owner arg       -> {"code": "invalid_arguments"}              PASS
6. numeric reference          -> {"code": "invalid_arguments"}              PASS
7. string instead of object   -> {"code": "invalid_call"}                   PASS

passed 7/7 | records unchanged: True | real tool invocations: 3


## Step 4 — Save the report and compare it with the expected output

The report uses the same schema as the course's `extensions/expected/tools_result.json`.

In [17]:
import os

result = {
    'mode': 'simulated_tool_protocol',
    'passed': passed,
    'total': len(CASES),
    'records_unchanged': records_unchanged,
    'cases': rows,
}

EXPECTED = {
    'mode': 'simulated_tool_protocol', 'passed': 7, 'total': 7, 'records_unchanged': True,
    'cases': [
        {'case': 1, 'result': {'code': 'ok', 'status': 'under_review'}, 'passed': True},
        {'case': 2, 'result': {'code': 'unavailable'}, 'passed': True},
        {'case': 3, 'result': {'code': 'unavailable'}, 'passed': True},
        {'case': 4, 'result': {'code': 'tool_not_allowed'}, 'passed': True},
        {'case': 5, 'result': {'code': 'invalid_arguments'}, 'passed': True},
        {'case': 6, 'result': {'code': 'invalid_arguments'}, 'passed': True},
        {'case': 7, 'result': {'code': 'invalid_call'}, 'passed': True},
    ],
}
print('matches expected tools_result.json:', result == EXPECTED)

os.makedirs('murshid_outputs', exist_ok=True)
with open('murshid_outputs/day2_tools_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(json.dumps(result, indent=2))

matches expected tools_result.json: True
{
  "mode": "simulated_tool_protocol",
  "passed": 7,
  "total": 7,
  "records_unchanged": true,
  "cases": [
    {
      "case": 1,
      "result": {
        "code": "ok",
        "status": "under_review"
      },
      "passed": true
    },
    {
      "case": 2,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 3,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 4,
      "result": {
        "code": "tool_not_allowed"
      },
      "passed": true
    },
    {
      "case": 5,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 6,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 7,
      "result": {
        "code": "invalid_call"
      },
      "passed": true
    }
  ]
}


## Reflection

### Why validation and authorization are different

**Validation** asks: *is this request well-formed?* It checks structure, types, allowed tool names and the format of the reference. It can be decided from the request alone. Case 6 (`1001` as a number) and case 7 (a plain string) fail here.

**Authorization** asks: *is this user allowed to see this specific record?* It can't be decided from the request alone. It needs a **trusted identity** from outside the call plus the record's owner. Case 2 is a **perfectly valid** request, so validation passes, yet it must still be refused because `demo-user-1` doesn't own `REF-1002`.

So a request can be valid and still unauthorized. Knowing a reference number doesn't prove ownership. Case 5 shows why identity must never come from the call: the caller added `"owner": "demo-user-2"` to try to claim another user's permission. The dispatcher rejects any extra argument, and it reads identity only from `trusted_user`.

### Design decisions
- **Allowlist instead of a blocklist:** only `check_status` runs, so new or unknown tools (like `delete_record`) are rejected by default.
- **Strict keys:** extra or missing fields are rejected, not silently ignored.
- **Same reply for missing and not-owned records (`unavailable`):** the reply doesn't reveal which references exist.
- **Structured refusals, not exceptions:** cases 2–7 are **correct refusals**, not crashes.
- **Read-only, verified:** a snapshot comparison confirms `RECORDS` is unchanged, and the call counter confirms the tool ran only for the 3 valid calls.

### Limitations
- The identity is a **synthetic demo fixture**, not real authentication.
- This is **simulated** structured tool calling. The calls are written by hand, and no model selects the tool. It is not native function calling from a hosted model provider.
- One tool and seven fixed cases. Passing them doesn't prove the dispatcher is complete or secure in production.
- All data is fictional. No real records, IDs or personal data are used.